In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import torch.nn as nn

import numpy as np
import matplotlib.pyplot as plt

import requests

Vamos a poner a interactuar los modelos Alice y Edgar para que generen texto en cadena de forma continua (sin buscar una conversación coherente o estilo chat), evaluando cómo se propaga y evoluciona el contexto estilístico de cada autor.

Conectaremos ambos modelos en secuencia: el texto generado por uno alimentará al otro de forma iterativa (en un intercambio no conversacional), manteniendo la traza de su respectivo contexto de entrenamiento.

In [2]:
# Tokenizador de EleutherAI
tokenizer = AutoTokenizer.from_pretrained('EleutherAI/gpt-neo-125m')
tokenizer.pad_token_id = tokenizer.encode(' ')[0]

# Cargar dos GPT-neo y enviarlos a la GPU
modelAlice = AutoModelForCausalLM.from_pretrained('EleutherAI/gpt-neo-125m')
modelEdgar = AutoModelForCausalLM.from_pretrained('EleutherAI/gpt-neo-125m')

# -> GPU
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
modelAlice = modelAlice.to(device)
modelEdgar = modelEdgar.to(device)

config.json:   0%|          | 0.00/1.01k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

[transformers] GPTNeoForCausalLM LOAD REPORT from: EleutherAI/gpt-neo-125m
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0...11}.attn.attention.masked_bias     | UNEXPECTED |  | 
transformer.h.{0, 2, 4, 6, 8, 10}.attn.attention.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

[transformers] GPTNeoForCausalLM LOAD REPORT from: EleutherAI/gpt-neo-125m
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0...11}.attn.attention.masked_bias     | UNEXPECTED |  | 
transformer.h.{0, 2, 4, 6, 8, 10}.attn.attention.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
# Aventuras de Alicia en el País de las Maravillas
text = requests.get('https://www.gutenberg.org/cache/epub/11/pg11.txt').text
aliceTokens = tokenizer.encode(text, return_tensors='pt')[0]

# Edgar Allan Poe
text = requests.get('https://www.gutenberg.org/cache/epub/2148/pg2148.txt').text
edgarTokens = tokenizer.encode(text, return_tensors='pt')[0]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (52948 > 2048). Running this sequence through the model will result in indexing errors


In [4]:
# Resumen
print(f'Alice in Wonderland has   {len(aliceTokens):7,} tokens.')
print(f'Edgar Allen Poe text has {len(edgarTokens):7,} tokens.')

Alice in Wonderland has    52,948 tokens.
Edgar Allen Poe text has 197,302 tokens.


In [5]:
# ALICE optimizador
optimizerAlice = torch.optim.AdamW(modelAlice.parameters(), lr=5e-5)

# EDGAR optimizador
optimizerEdgar = torch.optim.AdamW(modelEdgar.parameters(), lr=5e-5)

In [6]:
# parámetros de entrenamiento
seq_len = 256  # longitud máxima de secuencia
batch_size = 16
num_samples = 476

In [7]:
# inicializar pérdidas
lossAlice = np.zeros(num_samples)
lossEdgar = np.zeros(num_samples)

for sampli in range(num_samples):

  ### --- Ajuste fino de ALICE
  # obtener un lote de datos
  ix = torch.randint(len(aliceTokens) - seq_len, size=(batch_size,))
  X = aliceTokens[ix[:, None] + torch.arange(seq_len)].to(device)

  # paso hacia adelante y obtención de la pérdida
  modelAlice.zero_grad()
  outputs = modelAlice(X, labels=X)

  # retropropagación y almacenamiento de la pérdida
  outputs.loss.backward()
  optimizerAlice.step()
  lossAlice[sampli] = outputs.loss.item()
  ### ---------------------

  ### --- Ajuste fino de EDGAR
  # obtener un lote de datos
  ixEdg = torch.randint(len(edgarTokens) - seq_len, size=(batch_size,))
  XEdg = edgarTokens[ixEdg[:, None] + torch.arange(seq_len)].to(device)

  # paso hacia adelante y obtención de la pérdida
  modelEdgar.zero_grad()
  outputsEdg = modelEdgar(XEdg, labels=XEdg)

  # retropropagación y almacenamiento de la pérdida
  outputsEdg.loss.backward()
  optimizerEdgar.step()
  lossEdgar[sampli] = outputsEdg.loss.item()
  ### ---------------------

  # actualizar la visualización del progreso
  if sampli % 77 == 0:
    print(
        f'Muestra {sampli:4}/{num_samples}, pérdidas (Alice/Edgar):'
        f' {lossAlice[sampli]:.2f}/{lossEdgar[sampli]:.2f}'
    )

Muestra    0/476, pérdidas (Alice/Edgar): 2.60/2.74
Muestra   77/476, pérdidas (Alice/Edgar): 1.91/2.24
Muestra  154/476, pérdidas (Alice/Edgar): 1.23/2.07
Muestra  231/476, pérdidas (Alice/Edgar): 0.74/1.89
Muestra  308/476, pérdidas (Alice/Edgar): 0.38/1.85
Muestra  385/476, pérdidas (Alice/Edgar): 0.30/1.51
Muestra  462/476, pérdidas (Alice/Edgar): 0.21/1.52


In [8]:
# iniciar la conversación
outAlice = tokenizer.encode(
    'Hello, my name is Alice.', return_tensors='pt'
).to(device)
print('\n\n** Alice dice:\n', tokenizer.decode(outAlice[0].cpu()), end='')

for _ in range(5):

  # Turno de Edgar
  outEdgar = modelEdgar.generate(
      outAlice,
      max_new_tokens=50,
      do_sample=True,
      pad_token_id=tokenizer.pad_token_id,
  )
  print(
      f'\n\n** Edgar dice (conteo total de tokens: {len(outEdgar[0])}):\n',
      tokenizer.decode(outEdgar[0][len(outAlice[0]) :].cpu()),
      end='',
  )

  # Turno de Alice
  outAlice = modelAlice.generate(
      outEdgar,
      max_new_tokens=50,
      do_sample=True,
      pad_token_id=tokenizer.pad_token_id,
  )
  print(
      f"\n\n** Alice dice (conteo total de tokens: {len(outAlice[0])}):\n",
      tokenizer.decode(outAlice[0][len(outEdgar[0]) :].cpu()),
      end='',
  )

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.




** Alice dice:
 Hello, my name is Alice.

** Edgar dice (conteo total de tokens: 57):
  My real name is Marie-Louise. My real name is
      Marie-Sergie. My real name is also Marie-Hillel Gardé. My real name is
     

** Alice dice (conteo total de tokens: 107):
  the Cat.

 I’m the Cat!
 I’m the Cat!
I’m the Cat!
I’re you!
I’m I!


** Edgar dice (conteo total de tokens: 157):
 Take me!
On this rock!
I buried you!
I threw to rest my brother!
You were never more my own!

 The rock is not your own!
On the rock you

** Alice dice (conteo total de tokens: 207):
  will come again!

Oh my poor head, my poor little head!
What a dear life
The thoughtless little furchild
Would	'
Never left it
The minute it learnt the

** Edgar dice (conteo total de tokens: 257):
 
‘what’ it has learned!
Its thoughts are never
Never forgotten!
It will	not
 Never be forgotten!
Never be forgotten!
Never be _ever_
Never be

** Alice dice (conteo total de tokens: 307):
  _ever_

  I declare, yours bequeurs be—
   

---

### **Dinámica de Intercambio y Crecimiento del Contexto**

* **Mecánica del bucle:** El proceso comienza introduciendo la frase inicial `"Hello, my name is Alice"`. Esta se alimenta al modelo de Edgar, el cual genera 50 nuevos tokens adicionales, acumulando un total de 57 tokens en la secuencia.


* **Efecto de retroalimentación (Input-Output):** Esos 57 tokens pasan a ser la nueva entrada (*input*) para el modelo de Alice, que a su vez genera otros 50 tokens nuevos, elevando el total a 107 tokens. Este ciclo de rebote continúa de ida y vuelta iterativamente.


* **Ventana de contexto en expansión:** Dado que cada turno concatena la salida anterior, la longitud total del vector de tokens aumenta de manera acumulativa en cada paso (alcanzando los 507 tokens al finalizar el bucle). En la consola solo imprimimos los tokens recién generados de cada turno para evitar redundancias, pero el contexto histórico completo se mantiene y viaja en cada iteración.


* **Estructura de las ejecuciones:** El bucle se ejecuta 5 veces, lo que equivale a 10 generaciones en total (un par de intervenciones por cada iteración entre Edgar y Alice).



>  **Nota:** Al utilizar muestreo probabilístico (`do_sample=True`), la ejecución del código puede repetirse múltiples veces para obtener resultados creativos y divergentes.
>
>

---

### **Esto no es un chat**

No en el sentido tradicional. A diferencia de plataformas como ChatGPT, estos modelos **no han sido entrenados con instrucciones o refuerzo conversacional** para comprender los roles de dos agentes dialogando entre sí.

Lo que estamos presenciando no es una conversación lógica o coherente, sino un **ejercicio puro de completado de secuencias de texto**, donde cada modelo toma la narrativa anterior y la extiende basándose estrictamente en las probabilidades estadísticas de su propio entrenamiento literario.

In [9]:
#GPT usa tokens especiales para distinguir un usuarui de un chati como ,pm_start|>user, <|im_end|>, <|im_start|>assistant o equivalentes

Lo que hemos programado es, en realidad, muy cercano a cómo funcionan las cosas en ChatGPT: la gran diferencia radica en que se emplean más tokens especiales involucrados en el chat, los cuales le permiten al modelo distinguir con precisión la entrada del usuario de su propia respuesta.